# 17.2 Common two-piece and three-piece terms

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Simple piecewise-linear terms have a variety of uses in otherwise linear models. In
this section we present three cases: allowing limited violations of the constraints, analyzing
infeasibility, and representing costs for variables that are meaningful at negative as
well as positive levels.

**Penalty terms for "soft" constraints**

Linear programs most easily express "hard" constraints: that production must be at
least at a certain level, for example, or that resources used must not exceed those available
. Real situations are often not nearly so definite. Production and resource use may

```ampl
set ORIG;                        # origins
set DEST;                        # destinations
param supply {ORIG} >= 0;        # amounts available at origins
param demand {DEST} >= 0;        # amounts required at destinations
              check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];
param npiece {ORIG,DEST} integer >= 1;
param rate {i in ORIG, j in DEST, p in 1..npiece[i,j]}
>= if p = 1 then 0 else rate[i,j,p-1];
param limit {i in ORIG, j in DEST, p in 1..npiece[i,j]-1}
> if p = 1 then 0 else limit[i,j,p-1];
var Trans {ORIG,DEST} >= 0;      # units to be shipped
minimize Total_Cost:
       sum {i in ORIG, j in DEST}
              <<{p in 1..npiece[i,j]-1} limit[i,j,p];
              {p in 1..npiece[i,j]} rate[i,j,p]>> Trans[i,j];
subject to Supply {i in ORIG}:
       sum {j in DEST} Trans[i,j] = supply[i];
subject to Demand {j in DEST}:
       sum {i in ORIG} Trans[i,j] = demand[j];
```

**[Figure 17-3a](./17_1_cost_terms.ipynb#fig-17-3a):** Piecewise-linear model with indexed slopes (transpl2.mod).

have certain preferred levels, yet we may be allowed to violate these levels by accepting
some extra costs or reduced profits. The resulting "soft" constraints can be modeled by
adding piecewise-linear "penalty" terms to the objective function.

For an example, we return to the multi-week production model developed in [Chapter 4](../04/04.md).
As seen in [Figure 4-4](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-4), the constraints say that, in each of weeks 1 through T, total
hours used to make all products may not exceed hours available:

```ampl
subject to Time {t in 1..T}:
       sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];
```

Suppose that, in reality, a larger number of hours may be used in each week, but at some
penalty per hour to the total profit. Specifically, we replace the parameter avail[t] by
two availability levels and an hourly penalty rate:

```ampl
param avail_min {1..T} >= 0;
param avail_max {t in 1..T} >= avail_min[t];
param time_penalty {1..T} > 0;
```

Up to avail_min[t] hours are available without penalty in week t, and up to
avail_max[t] hours are available at a loss of time_penalty[t] in profit for each
hour above avail_min[t].

To model this situation, we introduce a new variable Use[t] to represent the hours
used by production. Clearly Use[t] may not be less than zero, or greater than

```ampl
param: ORIG: supply :=
       GARY 1400 CLEV 2600  PITT 2900 ;
param: DEST: demand :=
       FRA 900   DET 1200   LAN 600    WIN 400
       STL 1700  FRE 1100   LAF 1000 ;
param npiece: FRA DET LAN WIN STL FRE LAF :=
       GARY    3   3   3   2   3   2   3
       CLEV    3   3   3   3   3   3   3
       PITT    2   2   2   2   1   2   1 ;
param rate :=
[GARY,FRA,*] 1 39  2     50        3     70  [GARY,DET,*] 1 14  2  17  3  33
[GARY,LAN,*] 1 11  2     12        3     23  [GARY,WIN,*] 1 14  2  17
[GARY,STL,*] 1 16  2     23        3     40  [GARY,FRE,*] 1 82  2  98
[GARY,LAF,*] 1 8   2     16        3     24
[CLEV,FRA,*] 1 27  2     37        3     47  [CLEV,DET,*] 1 9   2 19  3 24
[CLEV,LAN,*] 1 12  2     32        3     39  [CLEV,WIN,*] 1 9   2 14  3 21
[CLEV,STL,*] 1 26  2     36        3     47  [CLEV,FRE,*] 1 95  2 105 3 129
[CLEV,LAF,*] 1 8   2     16        3     24
[PITT,FRA,*] 1 24 2 34  [PITT,DET,*] 1 14  2 24
[PITT,LAN,*] 1 17 2 27  [PITT,WIN,*] 1 13  2 23
[PITT,STL,*] 1 28       [PITT,FRE,*] 1 99  2 140
[PITT,LAF,*] 1 20 ;
param limit :=
[GARY,*,*] FRA 1    500     FRA        2    1000     DET 1 500   DET 2 1000
              LAN   1     500      LAN        2    1000     WIN 1 1000
              STL   1     500      STL        2    1000     FRE 1 1000
              LAF   1     500      LAF        2    1000
[CLEV,*,*] FRA  1    500     FRA        2    1000     DET 1   500  DET 2 1000
              LAN   1    500     LAN        2    1000     WIN 1   500  WIN 2 1000
              STL   1    500     STL        2    1000     FRE 1   500  FRE 2 1000
              LAF   1    500     LAF        2    1000
[PITT,*,*] FRA 1 1000 DET 1 1000  LAN 1 1000 WIN 1 1000
              FRE 1 1000 ;
```

**[Figure 17-3b](./17_1_cost_terms.ipynb#fig-17-3b):** Data for piecewise-linear model (transpl2.dat).

avail_max[t]. In place of our previous constraint, we say that the total hours used to
make all products must equal Use[t]:

```ampl
var Use {t in 1..T} >= 0, <= avail_max[t];
subject to Time {t in 1..T}:
       sum {p in PROD} (1/rate[p]) * Make[p,t] = Use[t];
```

We can now describe the hourly penalty in terms of this new variable. If Use[t] is
between 0 and avail_min[t], there is no penalty; if Use[t] is between
avail_min[t] and avail_max[t], the penalty is time_penalty[t] per hour

```ampl
penalty                           ← slope = time_penalty[t]

							    Use[t]
	  avail_min[t]
```

**[Figure 17-4](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-4):** Piecewise-linear penalty function for hours used.

that it exceeds avail_min[t]. That is, the penalty is a piecewise-linear function of
Use[t] as shown in [Figure 17-4](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-4), with slopes of 0 and time_penalty[t] surrounding
a breakpoint at avail_min[t]. Using the syntax previously introduced, we can
rewrite the expression for the objective function as:

```ampl
maximize Net_Profit:
       sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
       prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t])
       - sum {t in 1..T} <<avail_min[t]; 0,time_penalty[t]>> Use[t];
```

The first summation is the same expression for total profit as before, while the second is
the sum of the piecewise-linear penalty functions over all weeks. Between << and >> are
the breakpoint avail_min[t] and a list of the surrounding slopes, 0 and
time_penalty[t]; this is followed by the argument Use[t].

The complete revised model is shown in [Figure 17-5a](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5a), and our small data set from
[Chapter 4](../04/04.md) is expanded with the new availabilities and penalties in [Figure 17-5b](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5b). In the
optimal solution, we find that the hours used are as follows:

```ampl
ampl: model steelpl1.mod; data steelpl1.dat; solve;
MINOS 5.5: optimal solution found.

21 iterations, objective 457572.8571
ampl: display avail_min,Use,avail_max;
: avail_min Use avail_max     :=
1     35      35     42
2     35      42     42
3     30      30     40
4     35      42     42
;
```

In weeks 1 and 3 we use only the unpenalized hours available, while in weeks 2 and 4 we
also use the penalized hours. Solutions to piecewise-linear programs usually display this
sort of solution, in which many (though not necessarily all) of the variables "stick" at
one of the breakpoints.

```ampl
set PROD;                                # products
param T > 0;                             # number of weeks
param           rate {PROD} > 0;         #    tons per hour produced
param           inv0 {PROD} >= 0;        #    initial inventory
param           commit {PROD,1..T} >= 0; #    minimum tons sold in week
param           market {PROD,1..T} >= 0; #    limit on tons sold in week
param avail_min {1..T} >= 0;    # unpenalized hours available
param avail_max {t in 1..T} >= avail_min[t]; # total hours avail
param time_penalty {1..T} > 0;
param prodcost {PROD} >= 0;     # cost/ton produced
param invcost {PROD} >= 0;      # carrying cost/ton of inventory
param revenue {PROD,1..T} >= 0; # revenue/ton sold
var Make {PROD,1..T} >= 0;              # tons produced
var Inv {PROD,0..T} >= 0;               # tons inventoried
var Sell {p in PROD, t in 1..T}
       >= commit[p,t], <= market[p,t];      # tons sold
var Use {t in 1..T} >= 0, <= avail_max[t]; # hours used
maximize Total_Profit:
       sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
              prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t])
- sum {t in 1..T} <<avail_min[t]; 0,time_penalty[t]>> Use[t];
                                   # Objective: total revenue less costs in all weeks
subject to Time {t in 1..T}:
       sum {p in PROD} (1/rate[p]) * Make[p,t] = Use[t];
                                   # Total of hours used by all products
                                   # may not exceed hours available, in each week
subject to Init_Inv {p in PROD}: Inv[p,0] = inv0[p];
                                   # Initial inventory must equal given value
subject to Balance {p in PROD, t in 1..T}:
       Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];
                                   # Tons produced and taken from inventory
                                   # must equal tons sold and put into inventory
```

**[Figure 17-5a](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5a):** Piecewise-linear objective with penalty function (steelpl1.mod).

Dealing with infeasibility
The parameters commit[p,t] in [Figure 17-5b](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5b) represent the minimum production
amounts for each product in each week. If we change the data to raise these commitments:

```ampl
param commit:     1        2       3       4 :=
       bands   3500     5900    3900    6400
       coils   2500     2400    3400    4100 ;

param T := 4;

set PROD := bands coils;
param:         rate  inv0  prodcost invcost :=
       bands   200     10        10     2.5
       coils   140      0        11       3 ;
param:  avail_min   avail_max  time_penalty :=
       1       35          42          3100
       2       35          42          3000
       3       30          40          3700
       4       35          42          3100 ;
param revenue:      1     2     3     4 :=
       bands       25    26    27    27
       coils       30    35    37    39 ;
param commit:      1      2      3      4 :=
       bands    3000   3000   3000   3000
       coils    2000   2000   2000   2000 ;
param market:     1      2     3      4 :=
       bands   6000   6000  4000   6500
       coils   4000   2500  3500   4200
```

**[Figure 17-5b](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5b):** Data for [Figure 17-5a](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5a) (steelpl1.dat).

then there are not enough hours to produce even these minimum amounts, and the solver
reports that the problem is infeasible:

```ampl
ampl: model steelpl1.mod;
ampl: data steelpl2.dat;
ampl: solve;
MINOS 5.5: infeasible problem.

13 iterations
```

In the solution that is returned, the inventory of coils in the last period is negative:

```ampl
ampl: option display_1col 0;
ampl: display Inv;
Inv [*,*] (tr)
: bands   coils    :=
0   10        0
1    0      937
2    0      287
3    0        0
4    0    -2700
;
```

and production of coils in several periods is below the minimum required:

```ampl
← slope = time_penalty[t]
                   .

    penalty        .

                   .

                   .

                   .

                   .

                   .                        Use[t]
                             avail_min[t]                          avail_max[t]
```

**[Figure 17-6](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-6):** Penalty function for hours used, with two breakpoints.

```ampl
ampl:   display commit,Make,market;
:         commit   Make market    :=
bands   1   3500   3490   6000
bands   2   5900   5900   6000
bands   3   3900   3900   4000
bands   4   6400   6400   6500
coils   1   2500   3437   4000
coils   2   2400   1750   2500
coils   3   3400   2870   3500
coils   4   4100   1400   4200
;
```

These are typical of the infeasible results that solvers return. The infeasibilities are scattered
around the solution, so that it is hard to tell what changes might be necessary to
achieve feasibility. By extending the idea of penalties, we can better concentrate the
infeasibility where it can be understood.

Suppose that we want to view the infeasibility in terms of a shortage of hours. Imagine
that we extend the piecewise-linear penalty function of [Figure 17-4](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-4) to the one shown
in [Figure 17-6](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-6). Now Use[t] is allowed to increase past avail_max[t], but only
with an extremely steep penalty per hour — so that the solution will use hours above
avail_max[t] only to the extent absolutely necessary.

In AMPL, the new penalty function is introduced through the following changes:

```ampl
var Use {t in 1..T} >= 0;
maximize Total_Profit:
       sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
       prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t])
       - sum {t in 1..T} <<avail_min[t],avail_max[t];
                     0,time_penalty[t],100000>> Use[t];
```

The former bound avail_max[t] has become a breakpoint, and to its right a very
large slope of 100,000 has been introduced. Now we get a feasible solution, which uses
hours as follows:

```ampl
penalty
                                               Sell[p,t]

                            commit[p,t]
```

**[Figure 17-7](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-7):** Penalty function for sales.

```ampl
ampl: model steelpl2.mod; data steelpl2.dat; solve;
MINOS 5.5: optimal solution found.

19 iterations, objective -1576814.857
ampl: display avail_max,Use;
: avail_max Use     :=
1     42      42
2     42      42
3     40      41.7357
4     42      61.2857
;
```

This `table` implies that the commitments can be met only by adding about 21 hours,
mostly in the last week.

Alternatively, we may view the infeasibility in terms of an excess of commitments.
For this purpose we subtract a very large penalty from the objective for each unit that
Sell[p,t] falls below commit[p,t]; the penalty as a function of Sell[p,t] is
depicted in [Figure 17-7](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-7).

Since this function has a breakpoint at commit[p,t], with a slope of 0 to the right
and a very negative value to the left, it would seem that the AMPL representation could be

```ampl
<<commit[p,t]; -100000,0>> Sell[p,t]
```

Recall, however, AMPL's convention that such a function takes the value zero at zero.
[Figure 17-7](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-7) clearly shows that we want our penalty function to take a positive value at
zero, so that it will fall to zero at commit[p,t] and beyond. In fact we want the function
to take a value of 100000 * commit[p,t] at zero, and we could express the
function properly by adding this constant to the penalty expression:

```ampl
<<commit[p,t]; -100000,0>> Sell[p,t] + 100000*commit[p,t]
```

The same thing may be said more concisely by using a second argument that states
explicitly where the piecewise-linear function should evaluate to zero:

```ampl
<<commit[p,t]; -100000,0>> (Sell[p,t],commit[p,t])
```

This says that the function should be zero at commit[p,t], as [Figure 17-7](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-7) shows. In
the completed model, we have:

```ampl
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t];
maximize Total_Profit:
       sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
       prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t])
       - sum {t in 1..T} <<avail_min[t]; 0,time_penalty[t]>> Use[t]
       - sum {p in PROD, t in 1..T}
       <<commit[p,t]; -100000,0>> (Sell[p,t],commit[p,t]);
```

The rest of the model is the same as in [Figure 17-5a](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5a). Notice that Sell[p,t] appears in
both a linear and a piecewise-linear term within the objective function; AMPL automatically
recognizes that the sum of these terms is also piecewise-linear.

This version, using the same data, produces a solution in which the amounts sold are
as follows:

```ampl
ampl: model steelpl3.mod; data steelpl2.dat; solve;
MINOS 5.5: optimal solution found.

24 iterations, objective -293856347
ampl:   display Sell,commit;
:           Sell commit    :=
bands   1   3500   3500
bands   2   5900   5900
bands   3   3900   3900
bands   4   6400   6400
coils   1      0   2500
coils   2   2400   2400
coils   3   3400   3400
coils   4   3657   4100
;
```

To get by with the given number of hours, commitments to deliver coils are cut by 2500
tons in the first week and 443 tons in the fourth week.

**Reversible activities**

Almost all of the linear programs in this book are formulated in terms of nonnegative
variables. Sometimes a variable makes sense at negative as well as positive values, however,
and in many such cases the associated cost is piecewise-linear with a breakpoint at zero.

One example is provided by the inventory variables in [Figure 17-5a](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-5a). We have
defined Inv[p,t] to represent the tons of product `p` inventoried at the end of week t.
That is, after week `t` there are Inv[p,t] tons of product `p` that have been made but not
sold. A negative value of Inv[p,t] could thus reasonably be interpreted as representing
tons of product `p` that have been sold but not made — tons backordered, in effect.
The material balance constraints,

```ampl
subject to Balance {p in PROD, t in 1..T}:
       Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];
```

remain valid under this interpretation.
This analysis suggests that we remove the >= 0 from the declaration of Inv in our
model. Then backordering might be especially attractive if the sales price were expected
to `drop` in later weeks, like this:

```ampl
param revenue:    1    2    3   4 :=
     bands       25   26   23  20
     coils       30   35   31  25 ;
```

When we re-solve with appropriately revised model and data files, however, the results
are not what we expect:

```ampl
ampl: model steelpl4.mod; data steelpl4.dat; solve;
MINOS 5.5: optimal solution found.

15 iterations, objective 1194250
ampl:    display Make,Inv,Sell;
:          Make    Inv     Sell :=
bands    0     .       10    .
bands    1   0     -5990   6000
bands    2   0    -11990   6000
bands    3   0    -15990   4000
bands    4   0    -22490   6500
coils    0     .        0    .
coils    1   0     -4000   4000
coils    2   0     -6500   2500
coils    3   0    -10000   3500
coils    4   0    -14200   4200
;
```

The source of difficulty is in the objective function, where invcost[p] * Inv[p,t]
is subtracted from the sales revenue. When Inv[p,t] is negative, a negative amount is
subtracted, increasing the apparent total profit. The greater the amount backordered, the
more the total profit is increased — hence the odd solution in which the maximum possible
sales are backordered, while nothing is produced!

A proper inventory cost function for this model looks like the one graphed in [Figure 17-8](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-8).
It increases both as Inv[p,t] becomes more positive (greater inventories) and as
Inv[p,t] becomes more negative (greater backorders). We represent this piecewiselinear
function in AMPL by declaring a backorder cost to go with the inventory cost:

```ampl
Inv
      ← Backordered             Inventoried →
```

**[Figure 17-8](./17_2_common_twopiece_and_threepiece_terms.ipynb#fig-17-8):** Inventory cost function.

```ampl
param invcost {PROD} >= 0;
param backcost {PROD} >= 0;
```

Then the slopes for the Inv[p,t] term in the objective are -backcost[p] and
invcost[p], with the breakpoint at zero, and the correct objective function is:

```ampl
maximize Total_Profit:
       sum {p in PROD, t in 1..T}
       (revenue[p,t]*Sell[p,t] - prodcost[p]*Make[p,t]
       - <<0; -backcost[p],invcost[p]>> Inv[p,t])
       - sum {t in 1..T} <<avail_min[t]; 0,time_penalty[t]>> Use[t];
```

In contrast to our first example, the piecewise-linear function is subtracted rather than
added. The result is still piecewise-linear, though; it's the same as if we had added the
expression <<0; backcost[p], -invcost[p]>> Inv[p,t].

When we make this change, and add some backorder costs to the data, we get a more
reasonable-looking solution. Nevertheless, there remains a tendency to make nothing and
backorder everything in the later periods; this is an "end effect" that occurs because the
model does not account for the eventual production cost of items backordered past the
last period. As a quick `fix`, we can rule out any remaining backorders at the end, by
adding a constraint that final-week inventory must be nonnegative:

```ampl
subject to Final {p in PROD}:              Inv[p,T] >= 0;
```

Solving with this constraint, and with backcost values of 1.5 for band and 2 for coils:

```ampl
ampl: model steelpl5.mod; data steelpl5.dat; solve;
MINOS 5.5: optimal solution found.

20 iterations, objective 370752.8571
ampl:   display Make,Inv,Sell;
:            Make     Inv      Sell :=
bands   0      .        10       .
bands   1   4142.86       0  4152.86
bands   2   6000          0  6000
bands   3   3000          0  3000
bands   4   3000          0  3000
coils   0      .          0      .
coils   1   2000          0  2000
coils   2   1680      -820   2500
coils   3   2100      -800   2080
coils   4   2800          0  2000
;
```

About 800 tons of coils for weeks 2 and 3 will be delivered a week late under this plan.